In [125]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2 * oxygen + phosphorus, oxygen]

base_masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index(
    "nucleoside", drop=False
)
# dbg:
# masses = masses.loc[["A", "C", "G", "U"]]

In [126]:
def get_representatives(base_masses):
    # collapse nucleosides that have the same mass
    return (
        base_masses.groupby("monoisotopic_mass")
        .apply(lambda x: x.iloc[0])
        .set_index("nucleoside", drop=False)
    )


base_masses_representatives = get_representatives(base_masses)
base_masses_representatives

/tmp/ipykernel_13162/1235673937.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[0])


,nucleoside,monoisotopic_mass
nucleoside,,
C,C,243.08552
U,U,244.06954
A,A,267.09675
1A,1A,281.11240
19A,19A,282.09640
G,G,283.09167
67A,67A,295.09170
01A,01A,295.12810
019A,019A,296.11210


In [127]:
base_masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [128]:
import random, re
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CUA104GU1A")
print(true_sequence)
n_fragments = 20

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)


def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r


fragments = [random_fragment() for _ in range(n_fragments)]

true_fragment_masses = [
    sum(base_masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r])
    for l, r in fragments
]

fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]

start_fragments = [i for i in range(n_fragments) if fragments[i][0] == 0]
end_fragments = [i for i in range(n_fragments) if fragments[i][1] == len(true_sequence)]

fragments, fragment_masses, fragment_noise

['C', 'U', 'A', '104G', 'U', '1A']


([(0, 3),
  (5, 6),
  (2, 3),
  (2, 4),
  (2, 3),
  (4, 5),
  (0, 3),
  (2, 5),
  (2, 3),
  (3, 6),
  (5, 6),
  (1, 5),
  (4, 6),
  (2, 6),
  (1, 3),
  (0, 5),
  (4, 6),
  (3, 4),
  (0, 2),
  (3, 6)],
 [np.float64(754.5446282854934),
  np.float64(281.1195357661761),
  np.float64(267.08394158157427),
  np.float64(839.0821522481797),
  np.float64(267.0594372616425),
  np.float64(243.8598588254658),
  np.float64(754.5375550560406),
  np.float64(1082.1891310018477),
  np.float64(266.8760530320871),
  np.float64(1096.893499010391),
  np.float64(281.7059715256443),
  np.float64(1326.4097962774495),
  np.float64(524.5523922686699),
  np.float64(1363.4694544228407),
  np.float64(511.15598107229687),
  np.float64(1569.251963236259),
  np.float64(525.0605642256083),
  np.float64(571.2618932676211),
  np.float64(487.2183978538033),
  np.float64(1096.533042540443)],
 array([ 0.29281829,  0.00713577, -0.01280842,  0.77280225, -0.03731274,
        -0.20968117,  0.28574506, -0.189759  , -0.22069697, 

In [142]:
from itertools import combinations


def reduce_alphabet_by_fragments(the_base_masses, reduce_although_unexplained: bool):
    # TODO: the -2.0 should be informed by the variance in the measurements
    min_plausible_nucleoside_diff = the_base_masses["monoisotopic_mass"].min() - 2.0

    max_plausible_singular_nucleoside_mass = the_base_masses["monoisotopic_mass"].max() + 2.0

    start_masses = pd.Series(sorted(fragment_masses[i] for i in start_fragments))
    end_masses = pd.Series(sorted(fragment_masses[i] for i in end_fragments))

    def mass_diffs(masses):
        diffs = set(masses[1:].values - masses[:-1].values)
        return diffs
    
    def singular_masses(masses):
        return set(mass for mass in masses if mass <= max_plausible_singular_nucleoside_mass)

    diffs = sorted(mass_diffs(start_masses) | mass_diffs(end_masses) | singular_masses(fragment_masses))

    def is_similar(mass_a, mass_b):
        return abs(mass_a - mass_b) < 1.0

    def explain_diffs(diffs):
        explanations = {
            diff: [
                item.nucleoside
                for item in the_base_masses.itertuples()
                if is_similar(diff, item.monoisotopic_mass)
            ]
            for diff in diffs
        }
        explanations.update(
            {
                diff: [
                    (item_a.nucleoside, item_b.nucleoside)
                    for item_a, item_b in combinations(the_base_masses.itertuples(), 2)
                    if is_similar(
                        diff, item_a.monoisotopic_mass + item_b.monoisotopic_mass
                    )
                ]
                for diff in diffs
                if not explanations[diff]
            }
        )
        return explanations

    explanations = explain_diffs(diffs)

    # unexplained differences, only consider those
    unexplained = [
        diff
        for diff, expls in explanations.items()
        if not expls and diff > min_plausible_nucleoside_diff
    ]

    if unexplained and not reduce_although_unexplained:
        # at least one fragment differs by more than two nucleosides,
        # so we can't reduce the alphabet with this heuristic so far
        print("Unexplained differences:", unexplained)
        return None

    def get_nucleosides(explanations):
        for expls in explanations.values():
            for expl in expls:
                if isinstance(expl, tuple):
                    yield from expl
                else:
                    yield expl

    all_nucleosides = list(set(get_nucleosides(explanations)))

    return the_base_masses.loc[all_nucleosides]

In [143]:
reduced_alphabet_masses = reduce_alphabet_by_fragments(
    base_masses_representatives, reduce_although_unexplained=True
)
if reduced_alphabet_masses is None:
    reduced_alphabet_masses = base_masses_representatives
reduced_alphabet_masses

,nucleoside,monoisotopic_mass
nucleoside,,
61A,61A,335.15940
U,U,244.06954
1A,1A,281.11240
342G,342G,349.13860
A,A,267.09675
104G,104G,571.21260
42G,42G,335.12300
3483G,3483G,508.19180
00A,00A,479.10530


In [144]:
def estimate_seq(base_masses):
    prob = LpProblem("RNA sequencing", LpMinimize)
    # i = 1,...,n: positions in the sequence
    # j = 1,...,m: fragments
    # b = 1,...,k: (modified) bases

    n = len(true_sequence)
    m = len(fragment_masses)
    bases = base_masses.index.values

    # x: binary variables indicating fragment j presence at position i
    x = [
        [
            LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger)
            for j in range(m)
        ]
        for i in range(n)
    ]
    # y: binary variables indicating base at position i
    y = [
        {
            b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger)
            for b in bases
        }
        for i in range(n)
    ]
    # z: binary variables indicating product of x and y
    z = [
        [
            {
                b: LpVariable(f"z_{i},{j},{b}", lowBound=0, upBound=1, cat=LpInteger)
                for b in bases
            }
            for j in range(m)
        ]
        for i in range(n)
    ]
    # weight_diff_abs: absolute value of weight_diff
    predicted_mass_diff_abs = [
        LpVariable(f"predicted_mass_diff_abs_{j}", lowBound=0, cat=LpContinuous)
        for j in range(m)
    ]
    # weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
    preficted_mass_diff = [
        fragment_masses[j]
        - lpSum(
            [
                z[i][j][b] * base_masses.loc[b, "monoisotopic_mass"]
                for i in range(n)
                for b in bases
            ]
        )
        for j in range(m)
    ]

    # optimization function
    prob += lpSum([predicted_mass_diff_abs[j] for j in range(m)])

    # select one base per position
    for i in range(n):

        prob += lpSum([y[i][b] for b in bases]) == 1

    # fill z with the product of binary variables x and y
    for i in range(n):
        for j in range(m):
            for b in bases:
                prob += z[i][j][b] <= x[i][j]
                prob += z[i][j][b] <= y[i][b]
                prob += z[i][j][b] >= x[i][j] + y[i][b] - 1

    # ensure that fragment is aligned continuously
    # (no gaps: if x[i1,j] = 1 and x[i2,j] = 1, then x[i_between,j] = 1)
    for j in range(m):
        for i1, i2 in combinations(range(n), 2):
            # i2 and i1 are inclusive
            assert i2 > i1
            if i2 - i1 > 1:
                prob += (x[i1][j] + x[i2][j] - 1) * (i2 - i1 - 1) <= lpSum(
                    [x[i_between][j] for i_between in range(i1 + 1, i2)]
                )
                # for i_between in range(i1 + 1, i2):
                #     prob += x[i1][j] + x[i2][j] - 1 <= x[i_between][j]

    # ensure that start fragments are aligned at the beginning of the sequence
    for j in start_fragments:
        x[0][j].setInitialValue(1)
        x[0][j].fixValue()

    # ensure that end fragments are aligned at the end of the sequence
    for j in end_fragments:
        x[n - 1][j].setInitialValue(1)
        x[n - 1][j].fixValue()

    # ensure that inner fragments are neither aligned at the beginning or the end of the sequence
    for j in set(range(n_fragments)) - set(start_fragments) - set(end_fragments):
        x[0][j].setInitialValue(0)
        x[0][j].fixValue()
        x[n - 1][j].setInitialValue(0)
        x[n - 1][j].fixValue()

    # constrain weight_diff_abs to be the absolute value of weight_diff
    for j in range(m):
        prob += predicted_mass_diff_abs[j] >= preficted_mass_diff[j]
        prob += predicted_mass_diff_abs[j] >= -preficted_mass_diff[j]

    import pulp

    gurobi = pulp.getSolver("GUROBI_CMD")
    gurobi.msg = False
    result_ = prob.solve(gurobi)

    def get_base(i):
        for b in bases:
            if y[i][b].value() == 1:
                return b
        return None

    return [get_base(i) for i in range(n)], x, y, z, preficted_mass_diff

In [145]:
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np

reduce_to_k = 6


def reduce_alphabet(masses):
    kmeans = KMeans(n_clusters=reduce_to_k, random_state=213126)
    kmeans.fit(masses["monoisotopic_mass"].values.reshape(-1, 1))
    pseudo_nucs = [f"n{k}" for k in range(reduce_to_k)]
    nuc_mapping = defaultdict(list)
    for nuc, label in zip(masses["nucleoside"], kmeans.labels_):
        nuc_mapping[pseudo_nucs[label]].append(nuc)
    return (
        pd.DataFrame(
            {
                "nucleoside": pseudo_nucs,
                "monoisotopic_mass": kmeans.cluster_centers_[:, 0],
            }
        ).set_index("nucleoside", drop=False),
        nuc_mapping,
    )

In [146]:
def expand_alphabet(seq, nuc_mapping):
    return {nuc for pseudo_nuc in seq for nuc in nuc_mapping[pseudo_nuc]}

In [163]:
current_masses = reduced_alphabet_masses
predicted_sequence = None
last_masses_count = None

while True:
    print(f"remaining masses: {current_masses}")

    if False and (
        current_masses.shape[0] > reduce_to_k
        and current_masses.shape[0] != last_masses_count
    ):
        pseudo_masses, nuc_mapping = reduce_alphabet(current_masses)
        predicted_sequence, _, _, _, _ = estimate_seq(pseudo_masses)
        print(predicted_sequence, nuc_mapping)
    else:
        predicted_sequence, x, y, z, predicted_mass_diff = estimate_seq(current_masses)
        break

    last_masses_count = current_masses.shape[0]
    current_masses = current_masses.loc[
        list(expand_alphabet(predicted_sequence, nuc_mapping))
    ]
predicted_sequence

remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
61A               61A          335.15940
U                   U          244.06954
1A                 1A          281.11240
342G             342G          349.13860
A                   A          267.09675
104G             104G          571.21260
42G               42G          335.12300
3483G           3483G          508.19180
00A               00A          479.10530
34830G         34830G          524.18670
100G             100G          307.09170
3480G           3480G          466.18120
19A               19A          282.09640
C                   C          243.08552


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['C', 'U', 'A', '104G', 'U', '1A']

In [164]:
import altair as alt
import polars as pl

n = len(true_sequence)
m = len(fragment_masses)
fragment_coverage = pl.from_dicts(
    [
        {
            "l": min(i for i in range(n) if x[i][j].value() == 1) - 0.5,
            "r": max(i for i in range(n) if x[i][j].value() == 1) + 0.5,
            "j": j,
            "mass_info": f"{fragment_masses[j]:.2f} ({predicted_mass_diff[j].value():.2g})",
            "type": "predicted",
        }
        for j in range(m)
    ]
)

true_fragments = (
    pl.from_records(fragments)
    .transpose()
    .rename({"column_0": "l", "column_1": "r"})
    .with_columns(pl.col("r") - 1)
).with_columns(
    pl.col("l") - 0.5,
    pl.col("r") + 0.5,
    pl.arange(start=0, end=m).alias("j"),
    pl.Series(
        name="mass_info", values=[f"{mass:.2f}" for mass in true_fragment_masses]
    ),
    pl.lit("truth").alias("type"),
)
data = pl.concat([fragment_coverage, true_fragments])

In [165]:
seq_data = pl.DataFrame(
    {
        "nucleoside": true_sequence + predicted_sequence,
        "pos": list(range(len(true_sequence))) * 2,
        "type": ["truth"] * len(true_sequence)
        + ["predicted"] * len(predicted_sequence),
    }
)

base = alt.Chart(data)
(
    (
        base.mark_rule().encode(
            alt.X("l").axis(title=None, labels=False, ticks=False),
            alt.X2("r"),
            alt.Y("type").title(None),
        )
        + base.mark_text(align="left", dx=3).encode(
            alt.X("r").axis(title=None, labels=False, ticks=False),
            alt.Y("type").title(None),
            alt.Text("mass_info"),
        )
    ).facet(row=alt.Facet("j").title("fragments"))
    & alt.Chart(seq_data)
    .mark_text()
    .encode(
        alt.X("pos").title(None),
        alt.Y("type").title(None),
        alt.Text("nucleoside"),
        alt.Color("nucleoside", scale=alt.Scale(scheme="category10")).legend(None),
    )
).resolve_scale(x="shared")

alt.VConcatChart(...)

In [159]:
export_fragments = (
    pl.from_records(fragments)
    .transpose()
    .rename({"column_0": "l", "column_1": "r"})
    .with_columns(pl.col("r") - 1)
).with_columns(
    pl.Series(
        values=fragment_masses
    ).alias("observed_mass"),
    pl.Series(
        values=true_fragment_masses
    ).alias("true_mass"),
).rename({"l": "left", "r": "right"})
export_fragments.write_csv("/tmp/fragments.tsv", separator="\t")

In [166]:
start_fragments

[0, 6, 15, 18]